# GiftBridge Data Collection Prototype

This notebook is a coordinator-assisted, local pilot intake prototype. It captures consented item/need records without donor names, phone numbers, or addresses, validates values, stores entries locally, exports analysis files, and visualizes only the records collected here.

This is not the production GiftBridge PWA: it has no authentication, role enforcement, encrypted database, offline sync, or secure multi-user storage. Do not use it with sensitive or identifiable information. Synthetic fixtures remain separate from this notebook's `real` pilot entries.

In [ ]:
# 1. Set Up Project Structure
from pathlib import Path

cwd = Path.cwd()
WORKSPACE = next((path for path in (cwd, *cwd.parents) if (path / "ml" / "data").exists()), cwd)
COLLECTED_DIR = WORKSPACE / "data" / "collected"
EXPORT_DIR = WORKSPACE / "data" / "exports"
COLLECTED_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
DATA_FILE = COLLECTED_DIR / "pilot_intake.csv"
print(f"Workspace: {WORKSPACE}\nLocal pilot file: {DATA_FILE}")

In [ ]:
# 2. Import Required Libraries
import re
import tempfile
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import ipywidgets as widgets
from IPython.display import display

CATEGORIES = ["clothes", "books", "stationery", "toys", "utensils", "blankets", "electronics", "other"]
AGE_GROUPS = ["infant", "child", "teen", "adult", "all_ages", "unspecified"]
print("Libraries and GiftBridge controlled values loaded.")

In [ ]:
# 3. Design Data Collection Form (UI)
entry_type = widgets.Dropdown(options=[("Donation item", "item"), ("Village need", "need")], description="Record")
category = widgets.Dropdown(options=CATEGORIES, description="Category")
quantity = widgets.BoundedIntText(value=1, min=1, max=10000, description="Quantity")
age_group = widgets.Dropdown(options=AGE_GROUPS, value="unspecified", description="Age group")
condition = widgets.IntSlider(value=7, min=1, max=10, description="Condition")
urgency = widgets.IntSlider(value=3, min=1, max=5, description="Urgency")
village = widgets.Text(description="Village", placeholder="Required for a need")
notes = widgets.Textarea(description="Notes", placeholder="No names, phone numbers, or addresses", layout=widgets.Layout(width="80%", height="90px"))
consent = widgets.Checkbox(value=False, description="I have consent to record this anonymous pilot entry")
submit_button = widgets.Button(description="Validate and save", button_style="primary", icon="check")
form_output = widgets.Output()
form = widgets.VBox([entry_type, category, quantity, age_group, condition, urgency, village, notes, consent, submit_button, form_output])
display(form)


def _update_field_visibility(change=None):
    is_item = entry_type.value == "item"
    condition.layout.display = "flex" if is_item else "none"
    urgency.layout.display = "none" if is_item else "flex"
    village.layout.display = "none" if is_item else "flex"
    if change is not None:
        change.get("name")

entry_type.observe(_update_field_visibility, names="value")
_update_field_visibility()

In [ ]:
# 4. Implement Data Validation
def validate_entry(entry):
    errors = []
    if entry.get("record_type") not in {"item", "need"}:
        errors.append("Record type must be item or need.")
    if entry.get("category") not in CATEGORIES:
        errors.append("Choose a supported category.")
    try:
        if int(entry.get("quantity", 0)) < 1:
            errors.append("Quantity must be a positive whole number.")
    except (TypeError, ValueError):
        errors.append("Quantity must be a positive whole number.")
    if entry.get("age_group") not in AGE_GROUPS:
        errors.append("Choose a supported age group.")
    if entry.get("record_type") == "item":
        try:
            if not 1 <= int(entry.get("condition_score")) <= 10:
                errors.append("Condition must be between 1 and 10.")
        except (TypeError, ValueError):
            errors.append("Condition must be between 1 and 10.")
    if entry.get("record_type") == "need":
        if not str(entry.get("village", "")).strip():
            errors.append("Village is required for a need.")
        try:
            if not 1 <= int(entry.get("urgency")) <= 5:
                errors.append("Urgency must be between 1 and 5.")
        except (TypeError, ValueError):
            errors.append("Urgency must be between 1 and 5.")
    notes_text = str(entry.get("notes", ""))
    if len(notes_text) > 2000:
        errors.append("Notes must be 2,000 characters or fewer.")
    if re.search(r"[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}", notes_text, re.IGNORECASE):
        errors.append("Do not enter email addresses in notes.")
    if re.search(r"(?<!\d)(?:\+?91[\s-]?)?[6-9]\d{9}(?!\d)", notes_text):
        errors.append("Do not enter phone numbers in notes.")
    if not entry.get("consent", False):
        errors.append("Consent is required before saving a pilot entry.")
    return errors

print("Validation function ready: required fields, allowed ranges, consent, and basic PII patterns are checked.")

In [ ]:
# 5. Store Collected Data Locally
import os
import uuid


def append_record(record, path=DATA_FILE):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    new_rows = pd.DataFrame([record])
    if path.exists():
        existing = pd.read_csv(path, keep_default_na=False)
        combined = pd.concat([existing, new_rows], ignore_index=True)
    else:
        combined = new_rows
    with tempfile.NamedTemporaryFile(mode="w", suffix=".csv", dir=path.parent, delete=False, encoding="utf-8", newline="") as temporary:
        temporary_path = Path(temporary.name)
    try:
        combined.to_csv(temporary_path, index=False)
        os.replace(temporary_path, path)
    finally:
        temporary_path.unlink(missing_ok=True)
    return combined


def _save_form_entry(button):
    button.disabled = True
    entry = {
        "record_type": entry_type.value,
        "category": category.value,
        "quantity": quantity.value,
        "age_group": age_group.value,
        "condition_score": condition.value if entry_type.value == "item" else "",
        "urgency": urgency.value if entry_type.value == "need" else "",
        "village": village.value.strip() if entry_type.value == "need" else "",
        "notes": notes.value.strip(),
        "consent": consent.value,
    }
    errors = validate_entry(entry)
    with form_output:
        from IPython.display import clear_output
        clear_output(wait=True)
        if errors:
            print("Not saved:")
            print("\n".join(f"- {message}" for message in errors))
        else:
            entry.pop("consent")
            entry.update({
                "record_id": str(uuid.uuid4()),
                "data_origin": "real",
                "consent_confirmed": True,
                "created_at": datetime.now(timezone.utc).isoformat(),
            })
            saved = append_record(entry)
            print(f"Saved one consented entry. Local pilot rows: {len(saved)}")
            print("Keep this file private; it is not automatically uploaded or backed up.")
    button.disabled = False

submit_button.on_click(_save_form_entry)
print(f"Validated entries will be appended to {DATA_FILE}.")

In [ ]:
# 6. Export Data to CSV or Excel
def export_records(file_format="csv"):
    if not DATA_FILE.exists():
        raise FileNotFoundError("No pilot entries have been collected yet.")
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    if file_format == "csv":
        export_path = EXPORT_DIR / f"pilot-intake-{stamp}.csv"
        pd.read_csv(DATA_FILE, keep_default_na=False).to_csv(export_path, index=False)
    elif file_format == "xlsx":
        export_path = EXPORT_DIR / f"pilot-intake-{stamp}.xlsx"
        pd.read_csv(DATA_FILE, keep_default_na=False).to_excel(export_path, index=False)
    else:
        raise ValueError("file_format must be 'csv' or 'xlsx'")
    return export_path

print("Export helper ready. Run export_records('csv') after collecting entries; Excel export requires openpyxl.")

In [ ]:
# 7. Visualize Collected Data
import matplotlib.pyplot as plt

if DATA_FILE.exists():
    collected = pd.read_csv(DATA_FILE, keep_default_na=False)
    if collected.empty:
        print("No collected entries yet. Save a consented record to enable these charts.")
    else:
        counts = collected.groupby(["record_type", "category"], as_index=False).size()
        totals = collected.groupby(["record_type", "category"], as_index=False)["quantity"].sum()
        fig, axes = plt.subplots(1, 2, figsize=(13, 4))
        for record_kind, group in counts.groupby("record_type"):
            axes[0].bar(group["category"], group["size"], label=record_kind, alpha=0.8)
        axes[0].set_title("Collected records by category")
        axes[0].set_ylabel("Record count")
        axes[0].tick_params(axis="x", rotation=45)
        axes[0].legend(title="Record type")
        for record_kind, group in totals.groupby("record_type"):
            axes[1].bar(group["category"], group["quantity"], label=record_kind, alpha=0.8)
        axes[1].set_title("Reported quantity by category")
        axes[1].set_ylabel("Quantity")
        axes[1].tick_params(axis="x", rotation=45)
        axes[1].legend(title="Record type")
        plt.tight_layout()
        plt.show()
        print(f"Current local pilot file contains {len(collected)} consented entries (data_origin=real).")
else:
    print("No pilot entries have been collected. Synthetic fixtures are deliberately excluded from this view.")

In [ ]:
# 8. Add Unit Tests for Data Handling Functions
valid_item = {
    "record_type": "item", "category": "books", "quantity": 2,
    "age_group": "child", "condition_score": 8, "consent": True, "notes": "Clean books",
}
assert validate_entry(valid_item) == []
assert any("Village is required" in message for message in validate_entry({**valid_item, "record_type": "need"}))
assert any("phone numbers" in message for message in validate_entry({**valid_item, "notes": "Call 9876543210"}))

with tempfile.TemporaryDirectory() as temporary_dir:
    test_path = Path(temporary_dir) / "pilot.csv"
    first_write = append_record({"record_id": "test-1", "quantity": 2}, test_path)
    second_write = append_record({"record_id": "test-2", "quantity": 1}, test_path)
    assert len(first_write) == 1 and len(second_write) == 2
    assert pd.read_csv(test_path).record_id.tolist() == ["test-1", "test-2"]

print("Unit checks passed: valid entry, missing village, PII pattern rejection, and append-only CSV storage.")
if DATA_FILE.exists():
    collected_count = len(pd.read_csv(DATA_FILE, keep_default_na=False))
else:
    collected_count = 0
if collected_count:
    print(f"Conclusion: this local prototype currently contains {collected_count} consented pilot entries. These are local intake records, not yet validated study findings.")
else:
    print("Conclusion: the capture and storage path is ready, but there are no pilot records yet. No real-world supply-demand or impact conclusions can be drawn.")
print("Limitation: this notebook has no login, encrypted database, access control, offline synchronization, or production backup. Use the planned secured app for multi-user collection.")